In [1]:
from pymongo import AsyncMongoClient

# 连接

In [2]:
from config import DEFAULT_CONFIG

config = DEFAULT_CONFIG

In [3]:
mongodb_client = AsyncMongoClient(
    host=config.mongodb_uri
)

# 数据库

In [4]:
await mongodb_client.list_database_names()

['admin', 'config', 'dish_agent', 'local']

In [13]:
recipe_db = mongodb_client.get_database('recipe')

# 集合

In [5]:
recipe_db.create_collection('test_collection')

Collection(Database(MongoClient(host=['127.0.0.1:27017'], document_class=dict, tz_aware=False, connect=True), 'recipe'), 'test_collection')

In [6]:
recipe_db.list_collection_names()

['test_collection']

In [7]:
mongodb_client.list_database_names()

['admin', 'config', 'local', 'recipe']

In [8]:
test_collection = recipe_db.get_collection('test_collection')

# 插入文档

In [11]:
a = {
    'name': 'zyr', 
    'age': 18, 
    'love': {
        'food': ['薯片', '巧克力', 'beef'], 
        'moive': ['古墓丽影', '唐人街探案']
    }
}

In [12]:
test_collection.insert_one(a)

InsertOneResult(ObjectId('69e4e9efcea6fbe4ec24ccfd'), acknowledged=True)

In [13]:
test_collection.find_one({'name': 'zyr'})

{'_id': ObjectId('69e4e9efcea6fbe4ec24ccfd'),
 'name': 'zyr',
 'age': 18,
 'love': {'food': ['薯片', '巧克力', 'beef'], 'moive': ['古墓丽影', '唐人街探案']}}

In [14]:
import uuid

In [17]:
_id = str(uuid.uuid4())
_id

'3cede45c-1c7c-45f0-a841-dab356e535ea'

In [20]:
test_collection.insert_one(
    {
        'name': 'bq', 
        '_id': _id, 
        'love': 'yaru', 
    }
)

InsertOneResult('3cede45c-1c7c-45f0-a841-dab356e535ea', acknowledged=True)

# 更新

In [27]:
test_collection.update_one({'name':'bq'}, {'$set': {'sex': 'male'}})

UpdateResult({'n': 1, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

# 查找

In [21]:
test_collection.find_one({'_id': _id})

{'_id': '3cede45c-1c7c-45f0-a841-dab356e535ea', 'name': 'bq', 'love': 'yaru'}

In [25]:
for x in test_collection.find({}, {'name':1}):
    print(x)

{'_id': ObjectId('69e4e9efcea6fbe4ec24ccfd'), 'name': 'zyr'}
{'_id': '3cede45c-1c7c-45f0-a841-dab356e535ea', 'name': 'bq'}


In [30]:
for x in test_collection.find({'name':'zyr'}):
    print(x)

{'_id': ObjectId('69e4e9efcea6fbe4ec24ccfd'), 'name': 'zyr', 'age': 18, 'love': {'food': ['薯片', '巧克力', 'beef'], 'moive': ['古墓丽影', '唐人街探案']}}


# 构建食谱数据库

In [12]:
from rag_modules import DataPreparationModule
from config import DEFAULT_CONFIG

In [13]:
config = DEFAULT_CONFIG
data_module = await DataPreparationModule.create(config.data_path, config.mongodb_uri, config.mongodb_database, 
                                          config.documents_collection, config.collection_name)

data_module.load_documents();

In [50]:
len(data_module.documents)

323

In [9]:
data_module.documents[0]

Document(metadata={'source': '../../data/C8/cook/dishes/aquatic/水煮鱼.md', 'parent_id': 'ee8daa675045e06c89b9de679a9153c5', 'doc_type': 'parent', 'category': '水产', 'dish_name': '水煮鱼', 'difficulty': '困难'}, page_content='# 水煮鱼的做法\n\n水煮鱼是一道做法中等难度的硬菜。巴沙鱼富含优质蛋白且脂肪含量低，配合各种时令蔬菜十分营养健康。初学者一般需要 2 小时即可完成。\n\n预估烹饪难度：★★★★\n\n## 必备原料和工具\n\n- 巴沙鱼\n- 蔬菜（比如土豆片/豆芽/花菜/生菜/……）\n- 红油豆瓣酱\n- 藤椒油\n- 菜籽油\n- 白胡椒粉\n- 蒜瓣\n- 盐\n- 糖\n- 量杯\n- 厨房秤（可选）\n- 大不锈钢碗\n\n## 计算\n\n以下用量适合 3 至 5 人食用。\n\n- 巴沙鱼 500g\n- 蔬菜（比如土豆片/豆芽/花菜/生菜/……） 可有不同搭配，推荐合计重量 300g 至 500g\n- 红油豆瓣酱 40g （不怕辣想多加红油就多加 10 至 20g）\n- 豆豉 10g （可选）\n- 藤椒油 10ml\n- 菜籽油 25ml\n- 白胡椒粉 3g\n- 大蒜 2 瓣\n- 盐 5g\n- 糖 2g\n\n## 操作\n\n- 准备：巴沙鱼若是从冷冻柜里取出，需要放室温自然解冻 5 小时再做切片处理。\n- 切片：巴沙鱼撇成薄片，约 5cm 长，3cm 宽。\n- [腌制](../../tips/learn/学习腌.md)：将切好片的巴沙鱼放入大不锈钢碗中\n- 加入 30g 豆瓣酱，3g 盐，10ml 藤椒油，3g 白胡椒粉\n- 用手抓匀后加入 5ml 菜籽油收尾封住口味\n- 常温静置至少 30 分钟入味。\n- 备菜：大蒜切成蒜末。以 300g 花菜，200g 生菜为例，将花菜与生菜洗净。\n- 焯水与炒菜：花菜[开水锅焯水](../../tips/learn/学习焯水.md)备用；将生菜洗净晾干，炒熟备用（不用放油）。\n- 炒豆瓣酱：热锅冷油（菜籽油 20ml），加入 10g 豆瓣酱，10g 豆豉（可

In [11]:
dish_agent_db = mongodb_client.get_database('dish_agent')
recipe_collection = dish_agent_db.get_collection('recipe')

In [20]:
insert_items = [{
    '_id': doc.metadata['parent_id'], 
    'dish_name': doc.metadata['dish_name'], 
    'category': doc.metadata['category'], 
    'difficulty': doc.metadata['difficulty'], 
    'raw_content': doc.page_content
} for doc in data_module.documents]

In [23]:
insert_items[200]

{'_id': '0fb64ca95bf8fb43213d2194ce1e4448',
 'dish_name': '老式锅包肉',
 'category': '荤菜',
 'difficulty': '困难',
 'raw_content': '# 老式锅包肉的做法\n\n锅包肉是东北名菜，创始于光绪年间哈尔滨道台府厨师郑兴文之手。老式锅包肉的酸味来源于白醋汁，口味酸甜酥脆。\n\n预估烹饪难度：★★★★\n\n## 必备原料和工具\n\n- 猪通脊肉\n- 大葱\n- 姜\n- 蒜\n- 胡萝卜（可无）\n- 香菜\n- 白醋（建议使用 9 度的醋，这样才会有较为突出的老式锅包肉特有的醋香）\n- 白糖\n- 料酒\n- 盐\n- 味精\n- 土豆淀粉\n- 中筋面粉\n- 小苏打\n- 白熟芝麻（可无）\n- 食用油\n\n## 计算\n\n每份（约 2 人份）：\n\n- 猪通脊肉 300g\n- 大葱 50g\n- 姜 30g\n- 蒜 3-4 瓣\n- 胡萝卜 10g（可无）\n- 香菜 10g\n- 白熟芝麻 5g（可无）\n- 白醋 40g\n- 白糖 40g\n- 料酒 20ml\n- 盐 8g\n- 味精 5g\n- 米醋 5ml（可无）\n- 土豆淀粉 210g\n- 中筋面粉 70g\n- 小苏打 5g\n- 食用油 1000ml（用于炸制）\n\n## 操作\n\n1. **处理猪肉**：\n   - 将猪通脊肉切成厚度 8mm 的均匀肉片，去除白色筋膜。\n   - 用清水冲洗肉片，去除血水。\n   - 加入小苏打 5g，抓匀，静置 5 分钟。\n   - 用清水冲洗 1-2 次，去除多余小苏打。\n\n2. **腌制肉片**：\n   - 在肉片中加入盐 4g、料酒 5ml，拌匀，腌制 15 分钟。\n\n3. **准备挂浆**：\n   - **方法一**：\n     - 将土豆淀粉 100g 加入 200ml 清水，搅匀，静置 20 分钟。\n     - 倒出上层 2/3 的清水，保留底部淀粉浆，搅匀至酸奶状。\n   - **方法二（推荐）**：\n     - 将土豆淀粉 210g 和中筋面粉 70g 混合。\n     - 少量多次加入清水，搅拌至酸奶状，提起可拉丝，浆糊能在盆中堆积。\n     - 加入食用油 10m

In [24]:
recipe_collection.insert_many(insert_items)

InsertManyResult(['ee8daa675045e06c89b9de679a9153c5', '537a3610c19d1743ea41d710a900b180', '0a9c3ee7b037e6e4c304dfbe616c4554', '7830a7c1c1b3c31fb1d86709a00244ab', '0be38ce5979162867b20b0fb21495442', '336089f7f31f1ae8f106bb8ad219bb30', '74ce7655f477d8c91f22a170e30acd3a', '22a130e80e22c99f0a864de5e0dbe879', '64b7b0f32e23f7b6d311b7769754faa5', '7ebec7581c5a59f3fff597303f875387', '1a3866cba6451711114b7b84488edb55', 'dd36c17d00cf5694e1ab8bbbbfce4b7c', '2e894c040778c735574a2e7e04080f8e', '03556354495aad7636d85026b34b7bac', '90797a8368040ef053ced05f268dbfa4', '1043b1468828a1edc72058272809d624', '54cbb20f41af7e927dccec9b76c3b61f', '9f5434c18af5744128051a56cc69b14b', 'e484945890d3b390b72c5bf45e0f8f9d', 'e744a6dd3f64957eea7164445f0b1299', '26fe4424595f4f6cdb0ed33521fdec5f', 'ced3f64ef8fba43a5cb30683e057a306', '10c0bf51cf4fc317928320790eab0423', 'a8d62aca1273a4da20cab6ab4cf25c1d', '59a05b0b24775237ac071575c9aa7863', '14f46a19585e2ef679cbda1ef4a6b1b1', '7bd00ef157f82393e032a8d68d21b2c6', 'eddfdfa16

In [17]:
await recipe_collection.count_documents({})

323

In [43]:
async def find_many(*query: list[dict]):
    cursor = recipe_collection.find(*query)
    results = await cursor.to_list(length=None)
    return results

In [44]:
await find({'category': '早餐'}, {'dish_name': 1, 'category': 1})

{'category': '早餐'} {'dish_name': 1, 'category': 1}
{'_id': 'c1f30203089b43988b455fc95cf27704', 'dish_name': '空气炸锅面包片', 'category': '早餐'}
{'_id': 'a496f87f45b8968843c44e5e1a3a6604', 'dish_name': '微波炉蛋糕', 'category': '早餐'}
{'_id': 'd8b8353d0fc743a722368b29713903ce', 'dish_name': '煎饺', 'category': '早餐'}
{'_id': '15af6c96f9bf59881528ba71cc693854', 'dish_name': '太阳蛋', 'category': '早餐'}
{'_id': 'f39d422d3d6bc1e8883e61127fbbdbb0', 'dish_name': '金枪鱼酱三明治', 'category': '早餐'}
{'_id': 'd324bd120a25482dbe86a0f2c042fcd4', 'dish_name': '鸡蛋三明治', 'category': '早餐'}
{'_id': 'bca7bc938db63766eda7efc1c48a9703', 'dish_name': '牛奶燕麦', 'category': '早餐'}
{'_id': '4ff931bfc8ab4e68a87be951ee151666', 'dish_name': '茶叶蛋', 'category': '早餐'}
{'_id': '9f9f754b12fe546d05a49c0331a8ef06', 'dish_name': '桂圆红枣粥', 'category': '早餐'}
{'_id': 'a74dd12229c8081d9009c7f9b31f77e7', 'dish_name': '微波炉荷包蛋', 'category': '早餐'}
{'_id': '3d2eb79c834f0cbf90d09b895b70e947', 'dish_name': '水煮玉米', 'category': '早餐'}
{'_id': '97820b87a179c5da9d0c

In [71]:
data_module.chunk_documents()

[Document(metadata={'主标题': '水煮鱼的做法', 'source': '../../data/C8/cook/dishes/aquatic/水煮鱼.md', 'parent_id': 'ee8daa675045e06c89b9de679a9153c5', 'doc_type': 'child', 'category': '水产', 'dish_name': '水煮鱼', 'difficulty': '困难', 'chunk_id': '01f49c22c2dac3fafd940c5978fbc364', 'chunk_index': 0, 'batch_index': 0, 'chunk_size': 96}, page_content='水煮鱼 水产\n# 水煮鱼的做法  \n水煮鱼是一道做法中等难度的硬菜。巴沙鱼富含优质蛋白且脂肪含量低，配合各种时令蔬菜十分营养健康。初学者一般需要 2 小时即可完成。  \n预估烹饪难度：★★★★'),
 Document(metadata={'主标题': '水煮鱼的做法', '二级标题': '必备原料和工具', 'source': '../../data/C8/cook/dishes/aquatic/水煮鱼.md', 'parent_id': 'ee8daa675045e06c89b9de679a9153c5', 'doc_type': 'child', 'category': '水产', 'dish_name': '水煮鱼', 'difficulty': '困难', 'chunk_id': 'caf86ab6feca7026c541a701e1efa1ec', 'chunk_index': 1, 'batch_index': 1, 'chunk_size': 112}, page_content='水煮鱼 水产\n## 必备原料和工具  \n- 巴沙鱼\n- 蔬菜（比如土豆片/豆芽/花菜/生菜/……）\n- 红油豆瓣酱\n- 藤椒油\n- 菜籽油\n- 白胡椒粉\n- 蒜瓣\n- 盐\n- 糖\n- 量杯\n- 厨房秤（可选）\n- 大不锈钢碗'),
 Document(metadata={'主标题': '水煮鱼的做法', '二级标题': '计算', 'source': '../../data/C8/

In [73]:
data_module.chunks[0].metadata

{'主标题': '水煮鱼的做法',
 'source': '../../data/C8/cook/dishes/aquatic/水煮鱼.md',
 'parent_id': 'ee8daa675045e06c89b9de679a9153c5',
 'doc_type': 'child',
 'category': '水产',
 'dish_name': '水煮鱼',
 'difficulty': '困难',
 'chunk_id': '01f49c22c2dac3fafd940c5978fbc364',
 'chunk_index': 0,
 'batch_index': 0,
 'chunk_size': 96}

In [54]:
await find_one({'category': '早餐'})

{'_id': 'c1f30203089b43988b455fc95cf27704',
 'dish_name': '空气炸锅面包片',
 'category': '早餐',
 'difficulty': '非常简单',
 'raw_content': '# 空气炸锅面包片的做法\n\n健康饱肚子，适宜正在减脂期的程序员食用\n\n预估烹饪难度：★\n\n## 必备原料和工具\n\n- 面包片\n- 空气炸锅\n\n## 计算\n\n每份：\n\n- 面包片（两片）\n\n## 操作\n\n- 取出两片面包片（建议使用粗粮面包片）\n- 将面包片**垂直**放入空气炸锅\n- 200°C 烘烤 5 分钟\n- 取出即可使用\n\n## 附加内容\n\n营养成分表（数据基于全麦面包片）\n\n- 热量 254 千卡\n- 蛋白质 12.3 克\n- 脂肪 3.5 克\n- 碳水化合物 43.1 克\n- 膳食纤维 6.0 克'}

# 还原document

In [60]:
data_module.documents[0]

Document(metadata={'source': '../../data/C8/cook/dishes/aquatic/水煮鱼.md', 'parent_id': 'ee8daa675045e06c89b9de679a9153c5', 'doc_type': 'parent', 'category': '水产', 'dish_name': '水煮鱼', 'difficulty': '困难'}, page_content='# 水煮鱼的做法\n\n水煮鱼是一道做法中等难度的硬菜。巴沙鱼富含优质蛋白且脂肪含量低，配合各种时令蔬菜十分营养健康。初学者一般需要 2 小时即可完成。\n\n预估烹饪难度：★★★★\n\n## 必备原料和工具\n\n- 巴沙鱼\n- 蔬菜（比如土豆片/豆芽/花菜/生菜/……）\n- 红油豆瓣酱\n- 藤椒油\n- 菜籽油\n- 白胡椒粉\n- 蒜瓣\n- 盐\n- 糖\n- 量杯\n- 厨房秤（可选）\n- 大不锈钢碗\n\n## 计算\n\n以下用量适合 3 至 5 人食用。\n\n- 巴沙鱼 500g\n- 蔬菜（比如土豆片/豆芽/花菜/生菜/……） 可有不同搭配，推荐合计重量 300g 至 500g\n- 红油豆瓣酱 40g （不怕辣想多加红油就多加 10 至 20g）\n- 豆豉 10g （可选）\n- 藤椒油 10ml\n- 菜籽油 25ml\n- 白胡椒粉 3g\n- 大蒜 2 瓣\n- 盐 5g\n- 糖 2g\n\n## 操作\n\n- 准备：巴沙鱼若是从冷冻柜里取出，需要放室温自然解冻 5 小时再做切片处理。\n- 切片：巴沙鱼撇成薄片，约 5cm 长，3cm 宽。\n- [腌制](../../tips/learn/学习腌.md)：将切好片的巴沙鱼放入大不锈钢碗中\n- 加入 30g 豆瓣酱，3g 盐，10ml 藤椒油，3g 白胡椒粉\n- 用手抓匀后加入 5ml 菜籽油收尾封住口味\n- 常温静置至少 30 分钟入味。\n- 备菜：大蒜切成蒜末。以 300g 花菜，200g 生菜为例，将花菜与生菜洗净。\n- 焯水与炒菜：花菜[开水锅焯水](../../tips/learn/学习焯水.md)备用；将生菜洗净晾干，炒熟备用（不用放油）。\n- 炒豆瓣酱：热锅冷油（菜籽油 20ml），加入 10g 豆瓣酱，10g 豆豉（可

In [52]:
from langchain_core.documents import Document

In [61]:
item = await find_one({'dish_name': '水煮鱼'})

In [62]:
content = item.get('raw_content')
metadata = {}
for k, v in item.items():
    if k == '_id':
        metadata['parent_id'] = v
    elif k != 'raw_content':
        metadata[k] = v

In [63]:
Document(
    page_content=content, 
    metadata=metadata
)

Document(metadata={'parent_id': 'ee8daa675045e06c89b9de679a9153c5', 'dish_name': '水煮鱼', 'category': '水产', 'difficulty': '困难'}, page_content='# 水煮鱼的做法\n\n水煮鱼是一道做法中等难度的硬菜。巴沙鱼富含优质蛋白且脂肪含量低，配合各种时令蔬菜十分营养健康。初学者一般需要 2 小时即可完成。\n\n预估烹饪难度：★★★★\n\n## 必备原料和工具\n\n- 巴沙鱼\n- 蔬菜（比如土豆片/豆芽/花菜/生菜/……）\n- 红油豆瓣酱\n- 藤椒油\n- 菜籽油\n- 白胡椒粉\n- 蒜瓣\n- 盐\n- 糖\n- 量杯\n- 厨房秤（可选）\n- 大不锈钢碗\n\n## 计算\n\n以下用量适合 3 至 5 人食用。\n\n- 巴沙鱼 500g\n- 蔬菜（比如土豆片/豆芽/花菜/生菜/……） 可有不同搭配，推荐合计重量 300g 至 500g\n- 红油豆瓣酱 40g （不怕辣想多加红油就多加 10 至 20g）\n- 豆豉 10g （可选）\n- 藤椒油 10ml\n- 菜籽油 25ml\n- 白胡椒粉 3g\n- 大蒜 2 瓣\n- 盐 5g\n- 糖 2g\n\n## 操作\n\n- 准备：巴沙鱼若是从冷冻柜里取出，需要放室温自然解冻 5 小时再做切片处理。\n- 切片：巴沙鱼撇成薄片，约 5cm 长，3cm 宽。\n- [腌制](../../tips/learn/学习腌.md)：将切好片的巴沙鱼放入大不锈钢碗中\n- 加入 30g 豆瓣酱，3g 盐，10ml 藤椒油，3g 白胡椒粉\n- 用手抓匀后加入 5ml 菜籽油收尾封住口味\n- 常温静置至少 30 分钟入味。\n- 备菜：大蒜切成蒜末。以 300g 花菜，200g 生菜为例，将花菜与生菜洗净。\n- 焯水与炒菜：花菜[开水锅焯水](../../tips/learn/学习焯水.md)备用；将生菜洗净晾干，炒熟备用（不用放油）。\n- 炒豆瓣酱：热锅冷油（菜籽油 20ml），加入 10g 豆瓣酱，10g 豆豉（可选），加入蒜末，**中火**慢炒。\n- 汆鱼片：加入 150ml 热水，水很快开后加入腌制好的鱼片，轻轻翻动让鱼片在水中散开，加入 2g 盐和 2g 

In [64]:
from rag_modules import MyMongoDB

In [67]:
a = await MyMongoDB.create(uri, 'dish_agent')

In [70]:
await a.find_one('recipe', {'dish_name': '糖醋里脊'})

{'_id': 'dd36c17d00cf5694e1ab8bbbbfce4b7c',
 'dish_name': '糖醋里脊',
 'category': '荤菜',
 'difficulty': '困难',
 'raw_content': '# 糖醋里脊的做法\n\n糖醋里脊是中国经典传统名菜之一，该菜品以猪里脊肉为主材，配以面粉、淀粉、醋等佐料，酸甜可口，让人食欲大开；该菜品在陕菜、豫菜、浙菜、鲁菜、川菜、淮扬菜、粤菜、闽菜里均有此菜。\n\n预估烹饪难度：★★★★\n\n## 必备原料和工具\n\n- 里脊肉\n- 醋\n- 白糖\n- 淀粉\n- 鸡蛋\n- 生抽\n- 料酒\n- 蚝油\n- 番茄酱\n- 白胡椒粉\n- 盐\n\n## 计算\n\n每份：\n\n- 里脊肉 500g\n- 醋 10g\n- 白糖 30g\n- 淀粉 50g\n- 鸡蛋 50g\n- 生抽 10ml\n- 料酒 20g\n- 蚝油 10g\n- 番茄酱 30ml\n- 白胡椒粉 5g\n- 食盐 10g\n\n## 操作\n\n- 腌肉：将猪里脊肉先切厚片，用刀背拍一拍，把肉拍松一点。切成一个手指头粗的条，加料酒，生抽，蚝油，食盐，白胡椒粉，一个鸡蛋，将肉用手抓匀，腌制 20 分钟以上。\n- 调酱：番茄酱+10g 醋+30g 白糖+150ml 清水，搅拌至糖融化，备用。\n- 裹粉：先把粉全部裹好再来炸，这样在炸的时候就不会手忙脚乱。准备一个大碗，里面放淀粉，把每一根肉条都满满裹上淀粉。\n- 炸制：油温 160 摄氏度下里脊，可以拿一个干筷子放在油里面试一下，周围冒小泡就可以下锅。\n- 炸到表面微黄可以捞出，全程中火。然后等油温升高到 200 摄氏度，把里脊倒进去重新炸一次，只需 40 秒，表皮就会很脆，马上捞出。\n- 裹酱：另外拿一个锅，锅里放底油，把调好的酱汁倒进去，煮到冒泡，把炸好的里脊放进去，翻炒，让每一根都裹上酱汁。\n- 下炸好的里脊肉翻炒，关火盛出。\n\n## 附加内容\n\n- 里脊要多炸几遍，注意火候，否则达不到外焦里嫩的效果！\n- [下厨房](https://www.xiachufang.com/recipe/104483435/)\n- [百度百科](https://baike.baidu.com/item/%E7%B3%96%E9%86

In [79]:
await recipe_collection.distinct('category')

['主食', '其他', '早餐', '水产', '汤品', '甜品', '素菜', '荤菜', '调料', '饮品']

In [80]:
await recipe_collection.distinct('difficulty')

['中等', '困难', '简单', '非常困难', '非常简单']

In [5]:
db = mongodb_client.get_database(config.mongodb_database)

In [8]:
await db.list_collection_names()

['recipe', 'recipe_chunks']

In [10]:
await db['recipe_chunks'].count_documents({})

1464

In [14]:
data_module.chunk_documents()

[Document(metadata={'主标题': '水煮鱼的做法', 'source': '../../data/C8/cook/dishes/aquatic/水煮鱼.md', 'parent_id': 'ee8daa675045e06c89b9de679a9153c5', 'doc_type': 'child', 'category': '水产', 'dish_name': '水煮鱼', 'difficulty': '困难', 'chunk_id': '01f49c22c2dac3fafd940c5978fbc364', 'chunk_index': 0, 'batch_index': 0, 'chunk_size': 96}, page_content='水煮鱼 水产\n# 水煮鱼的做法  \n水煮鱼是一道做法中等难度的硬菜。巴沙鱼富含优质蛋白且脂肪含量低，配合各种时令蔬菜十分营养健康。初学者一般需要 2 小时即可完成。  \n预估烹饪难度：★★★★'),
 Document(metadata={'主标题': '水煮鱼的做法', '二级标题': '必备原料和工具', 'source': '../../data/C8/cook/dishes/aquatic/水煮鱼.md', 'parent_id': 'ee8daa675045e06c89b9de679a9153c5', 'doc_type': 'child', 'category': '水产', 'dish_name': '水煮鱼', 'difficulty': '困难', 'chunk_id': 'caf86ab6feca7026c541a701e1efa1ec', 'chunk_index': 1, 'batch_index': 1, 'chunk_size': 112}, page_content='水煮鱼 水产\n## 必备原料和工具  \n- 巴沙鱼\n- 蔬菜（比如土豆片/豆芽/花菜/生菜/……）\n- 红油豆瓣酱\n- 藤椒油\n- 菜籽油\n- 白胡椒粉\n- 蒜瓣\n- 盐\n- 糖\n- 量杯\n- 厨房秤（可选）\n- 大不锈钢碗'),
 Document(metadata={'主标题': '水煮鱼的做法', '二级标题': '计算', 'source': '../../data/C8/

In [15]:
len(data_module.chunks)

1619

In [17]:
chunk_ids = [chunk.metadata['chunk_id']for chunk in data_module.chunks]

In [18]:
len(chunk_ids)

1619

In [19]:
len(set(chunk_ids))

1614

In [20]:
a = set()
lis = []
for chunk_id in chunk_ids:
    if chunk_id in a:
        lis.append(chunk_id)
    else: a.add(chunk_id)

In [21]:
lis

['9517576d1a96112b2ea5350e9bc3e0c2',
 '3daa1f1792df59959a69085588e3a3bf',
 'e9ee65274cf8cace8b63c6e7389c6866',
 '473f8da964959e59e1a05c5a6e3f40be',
 'f293354100bdfbbf00e9a90482ff4320']

In [22]:
repeat_chunk = []
for chunk_id in lis:
    repeat_chunk.append([])
    for chunk in data_module.chunks:
        if chunk.metadata['chunk_id'] == chunk_id:
            repeat_chunk[-1].append(chunk)